In [1]:

from pathlib import Path
import csv
import json

def find_project_root(start: Path) -> Path:
    """Walk up from the current folder until we find the repo root."""
    for candidate in (start, *start.parents):
        if (candidate / "data" / "input").is_dir():
            return candidate
    return start


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "structured"
JSON_PATH = OUTPUT_DIR / "document_structure.json"
REPAIRED_DIR = OUTPUT_DIR / "reconstructed_tables"
MANIFEST_PATH = REPAIRED_DIR / "manifest.json"


def reconstruct_table(table, table_number):
    cells = table.get("data", {}).get("table_cells", [])

    if not cells:
        return None

    max_row = max(
        cell["start_row_offset_idx"] + cell.get("row_span", 1)
        for cell in cells
    )
    max_col = max(
        cell["start_col_offset_idx"] + cell.get("col_span", 1)
        for cell in cells
    )

    grid = [["" for _ in range(max_col)] for _ in range(max_row)]
    occupied = {}
    overlaps = []

    for cell in cells:
        row = cell["start_row_offset_idx"]
        col = cell["start_col_offset_idx"]
        row_span = cell.get("row_span", 1)
        col_span = cell.get("col_span", 1)
        text = cell.get("text", "").strip()

        # Record every grid position covered by this cell.
        for r in range(row, row + row_span):
            for c in range(col, col + col_span):
                if (r, c) in occupied:
                    overlaps.append({
                        "position": [r, c],
                        "existing_text": occupied[(r, c)],
                        "conflicting_text": text,
                    })
                else:
                    occupied[(r, c)] = text

        # Write the text only once, at the cell's starting position.
        grid[row][col] = text

    provenance = table.get("prov", [])
    page_number = provenance[0].get("page_no") if provenance else None

    merged_cells = [
        {
            "row": cell["start_row_offset_idx"],
            "column": cell["start_col_offset_idx"],
            "row_span": cell.get("row_span", 1),
            "column_span": cell.get("col_span", 1),
            "text": cell.get("text", ""),
        }
        for cell in cells
        if cell.get("row_span", 1) > 1 or cell.get("col_span", 1) > 1
    ]

    return {
        "table_number": table_number,
        "page": page_number,
        "rows": max_row,
        "columns": max_col,
        "grid": grid,
        "merged_cells": merged_cells,
        "overlaps": overlaps,
    }


def main():
    if not JSON_PATH.exists():
        raise FileNotFoundError(f"Missing input: {JSON_PATH}")

    data = json.loads(JSON_PATH.read_text(encoding="utf-8"))
    tables = data.get("tables", [])

    REPAIRED_DIR.mkdir(parents=True, exist_ok=True)
    manifest = []

    for number, table in enumerate(tables, start=1):
        result = reconstruct_table(table, number)

        if result is None:
            print(f"Table {number}: skipped; no cells found")
            continue

        csv_path = REPAIRED_DIR / f"table_{number:03}.csv"

        with csv_path.open(
            "w", newline="", encoding="utf-8-sig"
        ) as file:
            writer = csv.writer(file)
            writer.writerows(result["grid"])

        manifest.append({
            "table_number": number,
            "page": result["page"],
            "rows": result["rows"],
            "columns": result["columns"],
            "merged_cell_count": len(result["merged_cells"]),
            "overlap_count": len(result["overlaps"]),
            "merged_cells": result["merged_cells"],
            "overlaps": result["overlaps"],
            "csv_path": str(csv_path.relative_to(PROJECT_ROOT)),
        })

        print(
            f"Table {number}: page={result['page']}, "
            f"size={result['rows']}x{result['columns']}, "
            f"merged_cells={len(result['merged_cells'])}, "
            f"overlaps={len(result['overlaps'])}"
        )

    MANIFEST_PATH.write_text(
        json.dumps(manifest, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    print(f"\nReconstructed tables: {len(manifest)}")
    print(f"Manifest saved to: {MANIFEST_PATH}")


if __name__ == "__main__":
    main()


Table 1: page=1, size=2x2, merged_cells=0, overlaps=0
Table 2: page=1, size=23x4, merged_cells=11, overlaps=0
Table 3: page=1, size=14x4, merged_cells=0, overlaps=0
Table 4: page=2, size=19x6, merged_cells=0, overlaps=0
Table 5: page=2, size=7x7, merged_cells=3, overlaps=0
Table 6: page=2, size=18x6, merged_cells=1, overlaps=0
Table 7: page=3, size=22x6, merged_cells=0, overlaps=0
Table 8: page=3, size=16x6, merged_cells=0, overlaps=0
Table 9: page=3, size=23x6, merged_cells=0, overlaps=0
Table 10: page=3, size=23x6, merged_cells=0, overlaps=0
Table 11: page=4, size=9x3, merged_cells=0, overlaps=0
Table 12: page=4, size=6x4, merged_cells=0, overlaps=0

Reconstructed tables: 12
Manifest saved to: D:\Acalvio\DeCraft\outputs\structured\reconstructed_tables\manifest.json
